# Airline vs Oil Analysis (Yahoo Finance + yfinance)

This notebook analyzes the relationship between an airline stock (as proxy for a CFD underlying asset) and oil prices using real market data from Yahoo Finance.

- Code and comments are in English.
- Printed explanations are in Spanish.
- Default tickers:
  - Airline: `UAL`
  - Oil: `CL=F`


## 1) Installation and Imports


In [ ]:
# Optional: uncomment if yfinance is not installed in your environment
# %pip install yfinance

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
import yfinance as yf
from IPython.display import display

plt.style.use("seaborn-v0_8")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 10


: 

## 2) Input Parameters


In [ ]:
# Main parameters (easy to modify)
ticker_airline = "UAL"
ticker_oil = "CL=F"
start_date = "2020-01-01"
end_date = None  # None = use current date automatically
interval = "1d"  # Daily frequency

# Optional comparison set (extra section)
comparison_airlines = ["UAL", "DAL", "AAL"]


## 3) Data Download and Preparation


In [ ]:
def download_asset_data(ticker: str, start: str, end=None, interval: str = "1d") -> pd.DataFrame:
    # Download OHLCV data from Yahoo Finance and validate non-empty output.
    data = yf.download(
        ticker,
        start=start,
        end=end,
        interval=interval,
        auto_adjust=False,
        progress=False,
        threads=False,
    )

    if data is None or data.empty:
        raise ValueError(f"No se recibieron datos para {ticker}. Verifica ticker, fechas o conexión.")

    # Flatten possible MultiIndex columns from yfinance
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)

    return data


def get_preferred_price_series(df: pd.DataFrame, ticker: str) -> pd.Series:
    # Use Adj Close when available; otherwise fallback to Close.
    if "Adj Close" in df.columns:
        px = df["Adj Close"].copy()
        source_col = "Adj Close"
    elif "Close" in df.columns:
        px = df["Close"].copy()
        source_col = "Close"
    else:
        raise ValueError(f"No existe columna 'Adj Close' ni 'Close' para {ticker}.")

    px.name = ticker
    print(f"{ticker}: usando columna de precio '{source_col}'.")
    return px


def build_pair_dataframe(ticker_a: str, ticker_b: str, start: str, end=None, interval: str = "1d") -> pd.DataFrame:
    # Download both assets, align by date, and drop missing values.
    df_a = download_asset_data(ticker_a, start, end, interval)
    df_b = download_asset_data(ticker_b, start, end, interval)

    s_a = get_preferred_price_series(df_a, ticker_a)
    s_b = get_preferred_price_series(df_b, ticker_b)

    pair = pd.concat([s_a, s_b], axis=1)
    pair.index = pd.to_datetime(pair.index)
    pair = pair.sort_index().dropna()

    if pair.empty:
        raise ValueError("El cruce de fechas quedó vacío tras eliminar faltantes.")

    return pair


prices = build_pair_dataframe(ticker_airline, ticker_oil, start_date, end_date, interval)
print(f"Datos combinados: {prices.shape[0]} filas, desde {prices.index.min().date()} hasta {prices.index.max().date()}.")
display(prices.head())


## 4) Basic Visualization


In [ ]:
# Raw prices in separate subplots
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(prices.index, prices[ticker_airline], color="tab:blue", label=ticker_airline)
axes[0].set_title(f"{ticker_airline} - Price")
axes[0].set_ylabel("Price")
axes[0].legend()

axes[1].plot(prices.index, prices[ticker_oil], color="tab:orange", label=ticker_oil)
axes[1].set_title(f"{ticker_oil} - Price")
axes[1].set_ylabel("Price")
axes[1].set_xlabel("Date")
axes[1].legend()

plt.tight_layout()
plt.show()

# Normalized series to compare relative movement
normalized = prices / prices.iloc[0] * 100

plt.figure(figsize=(12, 5))
plt.plot(normalized.index, normalized[ticker_airline], label=f"{ticker_airline} (base=100)")
plt.plot(normalized.index, normalized[ticker_oil], label=f"{ticker_oil} (base=100)")
plt.title("Normalized Prices (Base 100)")
plt.xlabel("Date")
plt.ylabel("Indexed Level")
plt.legend()
plt.tight_layout()
plt.show()


## 5) Returns Transformation


In [ ]:
returns = pd.DataFrame(index=prices.index)
returns[f"{ticker_airline}_ret"] = prices[ticker_airline].pct_change()
returns[f"{ticker_oil}_ret"] = prices[ticker_oil].pct_change()
returns[f"{ticker_airline}_logret"] = np.log(prices[ticker_airline] / prices[ticker_airline].shift(1))
returns[f"{ticker_oil}_logret"] = np.log(prices[ticker_oil] / prices[ticker_oil].shift(1))
returns = returns.dropna()

print("Estadísticas descriptivas de retornos (simples y logarítmicos):")
display(returns.describe().T)


## 6) Correlation Analysis


In [ ]:
a_ret_col = f"{ticker_airline}_ret"
o_ret_col = f"{ticker_oil}_ret"

contemp_corr = returns[a_ret_col].corr(returns[o_ret_col])
rolling_30 = returns[a_ret_col].rolling(30).corr(returns[o_ret_col])
rolling_90 = returns[a_ret_col].rolling(90).corr(returns[o_ret_col])

print(f"Correlación contemporánea de retornos diarios: {contemp_corr:.4f}")

plt.figure(figsize=(12, 5))
plt.plot(rolling_30.index, rolling_30, label="Rolling corr 30d", alpha=0.9)
plt.plot(rolling_90.index, rolling_90, label="Rolling corr 90d", alpha=0.9)
plt.axhline(0, color="black", linestyle="--", linewidth=1)
plt.title("Rolling Correlation: Airline vs Oil Returns")
plt.xlabel("Date")
plt.ylabel("Correlation")
plt.legend()
plt.tight_layout()
plt.show()

print(
    "\nInterpretación (español):\n"
    "- Correlación positiva: ambos tienden a moverse en la misma dirección en esos periodos.\n"
    "- Correlación negativa: suelen moverse en direcciones opuestas.\n"
    "- Correlación inestable: la relación cambia con el tiempo, por lo que un único número promedio puede ocultar regímenes distintos."
)


## 7) Linear Regression (OLS)


In [ ]:
# OLS: airline returns = alpha + beta * oil returns
X = sm.add_constant(returns[o_ret_col])
y = returns[a_ret_col]
ols_model = sm.OLS(y, X).fit()

print(ols_model.summary())

alpha = ols_model.params["const"]
beta = ols_model.params[o_ret_col]
r2 = ols_model.rsquared
p_alpha = ols_model.pvalues["const"]
p_beta = ols_model.pvalues[o_ret_col]

direction = "positiva" if beta > 0 else "negativa"
significance = "estadísticamente significativa" if p_beta < 0.05 else "no estadísticamente significativa"

print(
    f"\nInterpretación (español):\n"
    f"- Beta estimada (sensibilidad al petróleo): {beta:.4f} ({direction}).\n"
    f"- Intercepto (alpha): {alpha:.6f}.\n"
    f"- R-cuadrado: {r2:.4f}.\n"
    f"- p-value beta: {p_beta:.4g} -> relación {significance} al 5%.\n"
    f"- p-value intercepto: {p_alpha:.4g}.\n"
    f"Conclusión parcial: la aerolínea muestra una sensibilidad {direction} frente al petróleo en promedio dentro de la muestra analizada."
)


## 8) Scatter Plot with Regression Line


In [ ]:
x = returns[o_ret_col]
y = returns[a_ret_col]

x_line = np.linspace(x.min(), x.max(), 200)
y_line = alpha + beta * x_line

plt.figure(figsize=(10, 6))
plt.scatter(x, y, alpha=0.4, s=18, label="Daily observations")
plt.plot(x_line, y_line, color="red", linewidth=2, label="OLS fit")
plt.title(f"{ticker_airline} Returns vs {ticker_oil} Returns")
plt.xlabel(f"{ticker_oil} daily return")
plt.ylabel(f"{ticker_airline} daily return")
plt.legend()

# Equation text box
text_eq = f"y = {alpha:.5f} + {beta:.3f}x"
plt.text(0.02, 0.95, text_eq, transform=plt.gca().transAxes,
         fontsize=11, verticalalignment="top",
         bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

plt.tight_layout()
plt.show()


## 9) Residual Analysis


In [ ]:
returns["residual"] = ols_model.resid
returns["residual_zscore"] = stats.zscore(returns["residual"], nan_policy="omit")

# Time series of residuals
plt.figure(figsize=(12, 4))
plt.plot(returns.index, returns["residual"], label="Residual")
plt.axhline(0, color="black", linestyle="--", linewidth=1)
plt.title("Residuals Over Time")
plt.xlabel("Date")
plt.ylabel("Residual")
plt.legend()
plt.tight_layout()
plt.show()

# Histogram
plt.figure(figsize=(10, 4))
plt.hist(returns["residual"], bins=50, alpha=0.75, color="tab:gray")
plt.title("Residual Distribution")
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

outliers = returns[np.abs(returns["residual_zscore"]) > 2].copy()

print(
    "Interpretación (español):\n"
    "Residuos con |z| > 2 pueden sugerir desvíos anormales entre la aerolínea y el petróleo "
    "respecto a la relación histórica estimada por el modelo."
)
print(f"\nCantidad de desvíos anormales detectados (|z| > 2): {len(outliers)}")
if not outliers.empty:
    display(outliers[[a_ret_col, o_ret_col, "residual", "residual_zscore"]].head(10))


## 10) Extreme Events


In [ ]:
# Top 10 absolute oil move days
extreme_days = returns.reindex(returns[o_ret_col].abs().sort_values(ascending=False).head(10).index).copy()
extreme_days = extreme_days.sort_index()

summary_cols = [o_ret_col, a_ret_col, "residual", "residual_zscore"]
extreme_summary = extreme_days[summary_cols].copy()
extreme_summary = extreme_summary.rename(columns={
    o_ret_col: "retorno_petroleo",
    a_ret_col: "retorno_aerolinea",
    "residual": "residuo",
    "residual_zscore": "zscore_residuo",
})

print("Top 10 días con mayor movimiento absoluto en petróleo y reacción de la aerolínea:")
display(extreme_summary)

print(
    "\nInterpretación (español): esta tabla ayuda a estudiar shocks energéticos o geopolíticos, "
    "comparando el movimiento extremo del petróleo con la respuesta de la aerolínea y el desvío del modelo."
)


## 11) Stress Windows Visualization


In [ ]:
stress_windows = [
    ("2022-02-01", "2022-04-30"),
    ("2023-10-01", "2023-11-30"),
]

for start_w, end_w in stress_windows:
    win_prices = prices.loc[start_w:end_w].copy()
    win_ret = returns.loc[start_w:end_w].copy()

    if win_prices.empty or win_ret.empty:
        print(f"Ventana {start_w} a {end_w}: sin datos suficientes.")
        continue

    win_norm = win_prices / win_prices.iloc[0] * 100

    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

    axes[0].plot(win_norm.index, win_norm[ticker_airline], label=f"{ticker_airline} (base=100)")
    axes[0].plot(win_norm.index, win_norm[ticker_oil], label=f"{ticker_oil} (base=100)")
    axes[0].set_title(f"Stress Window {start_w} to {end_w} - Normalized Prices")
    axes[0].set_ylabel("Indexed Level")
    axes[0].legend()

    axes[1].plot(win_ret.index, win_ret["residual"], label="Residual", color="tab:red")
    axes[1].axhline(0, color="black", linestyle="--", linewidth=1)
    axes[1].set_title("Model Residuals in Window")
    axes[1].set_ylabel("Residual")
    axes[1].set_xlabel("Date")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

print(
    "Estas ventanas pueden cambiarse manualmente para estudiar eventos geopolíticos específicos "
    "y ver si la relación histórica se mantiene o se rompe temporalmente."
)


## 12) Automatic Conclusion (Spanish)


In [ ]:
avg_30_corr = rolling_30.dropna().mean()
std_30_corr = rolling_30.dropna().std()

# Recent deviation check (last 30 observations)
recent = returns.tail(30)
recent_outliers = recent[np.abs(recent["residual_zscore"]) > 2]

if std_30_corr < 0.15:
    stability_label = "relativamente estable"
elif std_30_corr < 0.30:
    stability_label = "moderadamente variable"
else:
    stability_label = "inestable"

if recent_outliers.empty:
    recent_dev_msg = "No se observaron desvíos recientes relevantes (|z| > 2) en los últimos 30 días hábiles."
else:
    recent_dev_msg = (
        f"Se detectaron {len(recent_outliers)} desvíos recientes relevantes (|z| > 2) "
        "en los últimos 30 días hábiles."
    )

print("CONCLUSIÓN AUTOMÁTICA")
print("=" * 70)
print(f"Ticker aerolínea analizada: {ticker_airline}")
print(f"Ticker petróleo analizado: {ticker_oil}")
print(f"Correlación contemporánea promedio (muestra completa): {contemp_corr:.4f}")
print(f"Promedio de correlación móvil 30d: {avg_30_corr:.4f}")
print(f"Beta estimada frente al petróleo: {beta:.4f}")
print(f"Estabilidad de la relación (según variación de corr 30d): {stability_label}")
print(recent_dev_msg)
print(
    "Advertencia: correlación no implica causalidad. "
    "Además, un evento geopolítico puede romper una relación histórica observada."
)


## 13) Optional Multi-Airline Comparison (Extra)


In [ ]:
def compute_beta_vs_oil(airline_ticker: str, oil_ticker: str, start: str, end=None, interval: str = "1d"):
    # Return pair data, returns, correlation, and beta for one airline vs oil.
    pair_prices = build_pair_dataframe(airline_ticker, oil_ticker, start, end, interval)
    pair_ret = pair_prices.pct_change().dropna()

    if pair_ret.empty:
        raise ValueError(f"No hay retornos para {airline_ticker} vs {oil_ticker}.")

    x_col = oil_ticker
    y_col = airline_ticker

    X_local = sm.add_constant(pair_ret[x_col])
    y_local = pair_ret[y_col]
    model_local = sm.OLS(y_local, X_local).fit()

    return {
        "ticker": airline_ticker,
        "corr": pair_ret[y_col].corr(pair_ret[x_col]),
        "beta": model_local.params[x_col],
        "alpha": model_local.params["const"],
        "r2": model_local.rsquared,
        "p_beta": model_local.pvalues[x_col],
        "returns": pair_ret.rename(columns={y_col: f"{airline_ticker}_ret", x_col: f"{oil_ticker}_ret"}),
    }


results = []
returns_for_corr = []

for airline in comparison_airlines:
    try:
        res = compute_beta_vs_oil(airline, ticker_oil, start_date, end_date, interval)
        results.append(res)
        tmp = res["returns"][[f"{airline}_ret"]].copy()
        returns_for_corr.append(tmp)
    except Exception as e:
        print(f"No se pudo procesar {airline}: {e}")

if results:
    beta_table = pd.DataFrame([
        {
            "airline": r["ticker"],
            "corr_with_oil": r["corr"],
            "beta_vs_oil": r["beta"],
            "alpha": r["alpha"],
            "r_squared": r["r2"],
            "p_value_beta": r["p_beta"],
        }
        for r in results
    ]).sort_values("beta_vs_oil", ascending=False)

    print("Tabla final ordenada por beta estimada respecto al petróleo:")
    display(beta_table)

    # Correlation matrix among airline returns
    merged_airline_returns = pd.concat(returns_for_corr, axis=1).dropna()
    corr_matrix = merged_airline_returns.corr()

    print("Matriz de correlación de retornos entre aerolíneas:")
    display(corr_matrix)

    plt.figure(figsize=(8, 6))
    im = plt.imshow(corr_matrix.values, cmap="coolwarm", vmin=-1, vmax=1)
    plt.colorbar(im, fraction=0.046, pad=0.04)
    plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=45)
    plt.yticks(range(len(corr_matrix.index)), corr_matrix.index)
    plt.title("Airline Returns Correlation Matrix")
    plt.tight_layout()
    plt.show()
else:
    print("No se obtuvieron resultados en la comparación múltiple.")


## CFD Note (Important)

Este análisis usa el activo subyacente (acción de la aerolínea) como proxy del CFD.  
En la operativa real de CFDs, el resultado total también depende de costes adicionales como financiación overnight, spread y posibles ajustes del bróker.
